# Lesson 4

## Curating our model

In this lesson, we study **training stability** and **signal propagation** in neural networks.  
While the model architecture is the same as in the previous lesson, we will see *how signals and gradients flow through the network* and can strongly affect whether training succeeds or fails.

We will focus on:

- **diagnostics** to detect poor signal or gradient propagation (activation/gradient distributions, update ratios),
- the effect of **initialization scale** (avoiding saturation of the `tanh` activation),
- **Batch Normalization (BN)** in an MLP,
---

## Notation and shapes

- Vocabulary size: $V = 26 + 1$
- Context length: $T = 3$
- Embedding dimension: $d = 10$
- Hidden width: $H = 200$
- Batch size: $n$

Embeddings: $ \quad C \rightarrow E \rightarrow x \quad$ where $ \quad C \in \mathbb{R}^{V \times d}, \quad
E = C[X] \in \mathbb{R}^{n \times T \times d}, \quad
x = \mathrm{reshape}(E) \in \mathbb{R}^{n \times (T d)}. $


In [ ]:
## if running on google colab, run this cell
#%cd /content
#!rm -rf mini-course-DL
#!git clone https://github.com/AdamArras/mini-course-DL.git
#%cd mini-course-DL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch

## 1) Dataset and tools (reuse previous part)

**Exercise.**  
Rebuild the same data pipeline:
- load `names.txt`,
- build `stoi` / `itos`,
- set `block_size = T = 3`,
- implement `build_dataset(words)` producing `(X, Y)`,
- split into train / dev / test (80/10/10)

In [ ]:
# TODO:

## Tanh saturation

The first step in the MLP used in the previous lesson was as follow.

In [ ]:
d_emb      = 10
block_size = 3
H_hid      = 200
n_batch    = 32

batch = torch.randint(len(Xtr),(n_batch,))
Xb,Yb = Xtr[batch],Ytr[batch]

C = torch.randn(vocab_size,d_emb)
W0 = torch.randn(block_size * d_emb, H_hid)
b0 = torch.randn(H_hid)

E = C[Xb]
x = E.reshape(n_batch, -1)
h0 = torch.tanh(x @ W0 + b0)

In [ ]:
plt.subplot(211)
plt.hist(h0.flatten().numpy(), bins=100)
plt.subplot(212)
plt.imshow(abs(h0)>0.97)
plt.show()

We recall that $\tanh'(x) = 1 - \tanh^2(x)$. Why should the plot above alarm us about our algorithm?

By a variance calculation, show that the good normalisation should be $1/\sqrt{ dT }$ for the entries of $W_1 \in \mathbb{R}^{dT \times H}$ 

## 2) MLP: parameter initialization

We start with a **single hidden layer MLP** with normalised initialisation

Hyperparameters:
$$
d=10,\quad H=200,\quad n=32.
$$

Parameters:
$$
C\in\mathbb{R}^{V\times d},\quad
W_1\in\mathbb{R}^{(Td)\times H},\quad
W_2\in\mathbb{R}^{H\times V},\quad
b_2\in\mathbb{R}^{V}.
$$

### Initialization scale:

To avoid saturating $\tanh$, scale $W_1$ using a “gain”:
$$
W_1 \leftarrow \mathcal{N}(0,1)\cdot \frac{5}{3}\cdot \frac{1}{\sqrt{Td}}.
$$
Initialize:
$$
W_2 \leftarrow 0.01\cdot\mathcal{N}(0,1),\qquad b_2\leftarrow 0,
$$


**Exercise.**
Initialize all parameters with a fixed generator seed and set `requires_grad=True` on trainable parameters.



In [ ]:
# TODO: initialize C, W1, W2, b2, gamma, beta

## 3) Batch normalization

Given minibatch $X_b\in\mathbb{Z}^{n\times T}$:

1) Embedding + concat:
$$
x\in\mathbb{R}^{n\times (Td)}.
$$

2) Linear pre-activation:
$$
a = xW_1\in\mathbb{R}^{n\times H}.
$$

3) BatchNorm statistics (feature-wise over batch):
$$
\mu = \frac{1}{n}\sum_{k=1}^n a_k,\qquad
\sigma = \sqrt{\frac{1}{n-1}\sum_{k=1}^n (a_k-\mu)^2}.
$$

4) Normalize + affine:
$$
\tilde{a} = \gamma \odot \frac{a-\mu}{\sqrt{ \sigma^2 + \varepsilon}} + \beta
$$
The `eps` is here to ensure stability.

5) Nonlinearity:
$$
h = \tanh(\tilde{a}).
$$

6) Logits and loss:
$$
\text{logits} = hW_2 + b_2,\qquad
\mathcal{L}=\mathrm{CE}(\text{logits},Y_b).
$$

### Explanation

Batch Normalization normalizes activations to have zero mean and unit standard deviation **within each minibatch**.  
This improves training efficiency by stabilizing signal and gradient propagation.

Once training is finished, however, it is no longer appropriate to use statistics computed on the current evaluation batch.  
Instead, we want to normalize using statistics that reflect the **entire training set**.

For this purpose, we maintain two additional variables:
- a **running mean**,
- a **running standard deviation** (or variance).

These running statistics are updated **slowly during training** using moving averages of the batch statistics.  
They provide a smoothed estimate of the activation mean and standard deviation over the full training distribution.

Since these quantities are **not learned by backpropagation**, we must ensure that PyTorch does not track gradients when updating them (using `torch.no_grad()`), to avoid unnecessary memory usage.

Finally, we need a variable indicating whether the model is in **training mode** or **evaluation mode**, so that during the forward pass we can choose:
- batch mean and standard deviation during training,
- running mean and standard deviation during evaluation.

We need to add :
- BatchNorm affine parameters:
$$
\gamma\in\mathbb{R}^{1\times H},\quad \beta\in\mathbb{R}^{1\times H}.
$$

- Running buffers (that is, not trained by backprop):
$$
\mu_{\text{run}}\in\mathbb{R}^{1\times H},\quad
\sigma_{\text{run}}^2\in\mathbb{R}^{1\times H}.
$$

Q: why $n-1$ and not $n$ is the estimator $\sigma$ ? (compute the expectation over a random batch)

In [ ]:
# TODO: add BatchNorm in the parameters

**Exercise.**
Implement this forward pass exactly. Maintain exponential moving averages (EMA) of mean and std (or var):

$$\mu_{\text{run}} = 0.999 \mu_{\text{run}} + 0.001 \mu$$
$$\sigma_{\text{run}}^2 = 0.999 \sigma_{\text{run}}^2 + 0.001 \sigma^2$$

and:
$$\mu_{run}=\mathbf{0},\quad \sigma_{\text{run}}=\mathbf{1}$$
$$\gamma=\mathbf{1},\quad \beta=\mathbf{0},\quad$$

**Exercise.**
- Update running buffers under `torch.no_grad()`.
- Confirm no gradients are stored for running buffers.

In [ ]:
# TODO: forward pass with BN (training-mode)

## 5) Training loop (same SGD schedule)

Train for `max_steps = 200000` with minibatch size $n=32$.

Set the learning rate to be :
$$
\eta =
\begin{cases}
0.1 & i<100000,\\
0.01 & i\ge 100000.
\end{cases}
$$

**Exercise.**
Implement the training loop:
- sample minibatch,
- forward with BN training stats,
- backward,
- update trainable parameters.

Track and plot `loss.log10()` over time.

In [ ]:
# TODO: BN-MLP training loop

## 6) Evaluation with BN (eval mode)

At evaluation time, we do **not** use batch statistics; use running buffers:

$$
\tilde{a} = \gamma \odot \frac{a-\mu_{\text{run}}}{\sigma_{\text{run}}+\varepsilon} + \beta.
$$

**Exercise.**
Write `split_loss(split)` for train/dev/test that:
- runs forward on the full split,
- uses $\mu_{\text{run}},\sigma_{\text{run}}$,
- returns/prints cross-entropy.



In [ ]:
# TODO: evaluate train and dev losses

## 7) (Optional) Calibrate BN using full training set

Sometimes you compute BN stats on the entire training set after training:
$$
\mu_{\text{train}} = \mathbb{E}[a],\qquad \sigma_{\text{train}}=\mathrm{Std}(a)
$$
and use those for eval.

**Exercise.**
Compute these under `torch.no_grad()` and compare with running estimates.



In [ ]:
# TODO: optional BN calibration over full training set


## 8) Pytorchifying: implement minimal layer classes

We will implement tiny classes with an `__call__` forward and a `.parameters()` method.

### 8.1 Linear

$$
\mathrm{Linear}(x) = xW + b
$$
Initialize:
$$
W\sim \mathcal{N}(0,1)/\sqrt{\text{fan\_in}},\quad b=0.
$$

### 8.2 BatchNorm1d

During training:
$$
\mu=\mathrm{mean}(x),\quad v=\mathrm{var}(x),
\quad \hat{x}=\frac{x-\mu}{\sqrt{v+\varepsilon}},
\quad y=\gamma\odot \hat{x}+\beta.
$$
Update running buffers with momentum $m$:
$$
\mu_{\text{run}}\leftarrow (1-m)\mu_{\text{run}} + m\mu,
\quad
v_{\text{run}}\leftarrow (1-m)v_{\text{run}} + m v.
$$
During eval: use running buffers.

**Exercise.**
Implement the three classes:
- `Linear(fan_in, fan_out, bias=True/False)`
- `BatchNorm1d(dim, eps=1e-5, momentum=0.1)` with `.training` flag
- `Tanh()`

Each stores `self.out` on forward.



It might be usefull to read the documentation of Linear class in Pytorch, before implementing this class ourself. Here is the [link](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) 

In [ ]:
# TODO: implement Linear class

In [ ]:
# TODO: implement BatchNorm1d class

In [ ]:
# TODO: implement Tanh class


## 9) Build a deeper network from layers

Architecture:

$$
\underbrace{\big(\mathrm{Linear}\to \mathrm{BN}\to \tanh\big)\times 5}_{\text{hidden stack}}
\to
\mathrm{Linear}\to \mathrm{BN}.
$$

Example (with `bias=False` for linears):
- input dim: $Td$
- hidden dim: $H=100$
- output dim: $V$

**Exercise.**
1. Create embedding table $C$.
2. Create `layers = [...]` list of your layer objects.
3. Forward pass:
   - embed + concat to get $x\in\mathbb{R}^{n\times (Td)}$,
   - loop `x = layer(x)` through all layers,
   - treat final `x` as logits.
4. Loss = `F.cross_entropy(logits, Yb)`.



In [ ]:
# TODO: define C and layers list for a deep net.


In [ ]:
# TODO: forward pass through deep layers

## 10) Initialization tweaks for stable training

BatchNorm normalizes activations so that each feature has approximately
zero mean and unit variance. In the last layer this implies that, at
initialization,

$$
\hat{x} \sim \mathcal{N}(0,1),
$$

and therefore the logits after the affine transform

$$
z = \gamma \hat{x} + \beta
$$

typically have magnitude of order $1$.

In a softmax classifier we exponentiate these logits:
$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

If $z_i \sim \mathcal{N}(0,1)$, then $e^{z_i}$ follows a **log-normal distribution**, which is highly skewed. Even moderate Gaussian logits therefore produce exponentiated weights with large multiplicative fluctuations, so the softmax probabilities can deviate noticeably from the uniform distribution at initialization.

Ideally the model should start with predictions close to **uniform**, so that the first training steps do not arbitrarily favor some classes. However, we cannot initialize exactly at the uniform distribution because we still need **randomness to break symmetry**.

To balance these two goals, we shrink the gain of the final BatchNorm layer so that the initial logits are small:

$$
\gamma_{\text{last}} \leftarrow 0.1\,\gamma_{\text{last}}.
$$

This keeps the softmax closer to uniform while preserving random variations that allow learning to begin.

**Exercise.**  
Under `torch.no_grad()`, modify the parameters accordingly.

In [ ]:
# TODO: apply init tweaks (scale last gamma etc.)

## 11) Training loop for deep network + diagnostics

We now train the deeper network as before, but this time we also monitor
**signal propagation** through the network. Poor signal propagation is one
of the main reasons deep networks fail to train.

We will track three diagnostics during training.

### 11.1 Activation saturation (tanh)

For each `Tanh` layer, inspect the distribution of activations.  
Because `tanh` saturates near $-1$ and $1$, many activations in this
region indicate that gradients may vanish.

A simple diagnostic is the **fraction of saturated activations**, e.g.

$$
\text{sat} =
\frac{\#\{|a| > 0.97\}}{\text{total activations}}.
$$

If this fraction becomes large, the network is operating in a regime
where gradients may propagate poorly.

---

### 11.2 Gradient distributions

After `loss.backward()`, inspect the gradients flowing through the
network.

For example, for each `Tanh` layer you can examine `layer.out.grad` and track statistics such as the mean or standard deviation.  
Large differences between layers may indicate exploding or vanishing gradients.

---

### 11.3 Update-to-parameter ratio

For each parameter $p$, track the relative size of the update:

$$
\log_{10}\!\left(
\frac{\mathrm{Std}(\eta \nabla p)}{\mathrm{Std}(p)}
\right).
$$

This measures how large the parameter update is compared to the
magnitude of the parameter itself.

A useful heuristic is that this value should typically be around $ \approx -3 $ meaning that updates are roughly $10^{-3}$ times the parameter scale.

---

### Exercise

1. Implement training for many steps (the course uses $200\,000$; you may start with $1\,000$).
2. Track the loss during training.
3. Track the update-to-parameter ratios for weight matrices.

Plot:

- the **training loss curve**,
- the **update-ratio curves** for the different layers.

In [ ]:
# TODO: Implement training 

In [ ]:
# TODO: plot training loss curve, update-ratio curves.

## 12) Eval mode for deep network + sampling

Before evaluation, set:
```python
for layer in layers:
    layer.training = False
```

Then evaluate train/dev loss.

Sampling works the same way as in the previous lessons.

**Exercise.**
- Evaluate losses in eval mode.
- Sample ~20 names.

In [ ]:
# TODO: set eval mode and compute split losses


In [ ]:
# TODO: sample names from the deep network

## Checklist (when you're done)

You should be able to explain:

1. Why bad initialization saturates tanh and kills gradients.
2. What BN normalizes (per-feature across batch) and why that helps.
3. The difference between training-time BN and eval-time BN.
4. Why deep networks need diagnostics
5. How scaling the last layer reduces overconfident logits early in training.

**Open questions:**

- in the computation of the variance, I used `correction=0`, which computes the variance with the normalisation $1/(N-correction) = 1/N$ instead of $1/(N-1)$, so I use a biased estimator. Can you explain why?